# Tune Logistic Regression on Untouched and Retrained MPNet SentenceTransformer

This notebook loads both saved encoders, computes embeddings for the shared split, tunes One-vs-Rest Logistic Regression on validation macro F1, tunes per-aspect thresholds on validation predictions, and saves one classifier bundle per encoder.

Run `01-mpnet_retraining.ipynb` first.


In [1]:
from pathlib import Path
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_config
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid
from sentence_transformers import SentenceTransformer
from sklearn.multiclass import OneVsRestClassifier


d:\ZB\Code\UIT\CS221\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = Path.cwd().resolve()
for _ in range(8):
    if (path / "data" / "Restaurant_ABSA_processed.csv").exists():
        PROJECT_ROOT = path
        break
    path = path.parent
else:
    raise FileNotFoundError("Could not locate the project root")

DATA_PATH = PROJECT_ROOT / "data" / "Restaurant_ABSA_processed.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "MPNet"
MODEL_DIR = PROJECT_ROOT / "models" / "mpnet"
SPLIT_PATH = OUTPUT_DIR / "data_split.npz"

ENCODER_PATHS = {
    "unretrained": MODEL_DIR / "unretrained_encoder",
    "retrained": MODEL_DIR / "retrained_encoder",
}

if not SPLIT_PATH.exists():
    raise FileNotFoundError("Run 01-mpnet_retraining.ipynb before this notebook")

for name, encoder_path in ENCODER_PATHS.items():
    if not encoder_path.exists():
        raise FileNotFoundError(f"Missing {name} encoder: {encoder_path}")


In [3]:
if PROJECT_ROOT not in sys.path: 
    sys.path.insert(0, str(PROJECT_ROOT))
    
from manual_ovr_logistic_regression import ManualLogisticRegression

In [4]:
ASPECT_COLS = ["food", "price", "service", "ambiance", "miscellaneous"]
TEXT_COL = "review_cleaned"

df = pd.read_csv(DATA_PATH).dropna(subset=[TEXT_COL]).reset_index(drop=True)
texts = df[TEXT_COL].astype(str).to_numpy()
labels = df[ASPECT_COLS].to_numpy(dtype=np.int8)

split = np.load(SPLIT_PATH)
train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

X_train_text = texts[train_idx].tolist()
X_val_text = texts[val_idx].tolist()
y_train = labels[train_idx]
y_val = labels[val_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Held-out test:", len(test_idx))


Train: 477
Validation: 159
Held-out test: 160


In [5]:
GRID = {
    "C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
    "lr": [0.01, 0.001]
}

N_JOBS = min(4, os.cpu_count() or 1)
NORMALIZE_EMBEDDINGS = True
BATCH_SIZE = 32


In [6]:
def fit_one(params, X_train, X_val, y_train, y_val):
    model = OneVsRestClassifier(
        ManualLogisticRegression(
            C=params["C"],
            class_weight=params["class_weight"],
            lr=params["lr"],
            max_iter=3000,
            random_state=42,
        ),
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_val)
    predictions = (probabilities >= 0.5).astype(np.int8)
    per_class = f1_score(
        y_val,
        predictions,
        average=None,
        zero_division=0,
    )

    result = {
        **params,
        "macro_f1": f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "micro_f1": f1_score(
            y_val,
            predictions,
            average="micro",
            zero_division=0,
        ),
    }
    for index, aspect in enumerate(ASPECT_COLS):
        result[f"{aspect}_f1"] = per_class[index]
    return result


def run_grid_search(X_train, X_val):
    parameter_grid = list(ParameterGrid(GRID))
    with parallel_config(
        backend="loky",
        n_jobs=N_JOBS,
        inner_max_num_threads=1,
    ):
        results = Parallel(verbose=10, batch_size=1)(
            delayed(fit_one)(
                params,
                X_train,
                X_val,
                y_train,
                y_val,
            )
            for params in parameter_grid
        )

    return (
        pd.DataFrame(results)
        .sort_values(["macro_f1", "micro_f1"], ascending=False)
        .reset_index(drop=True)
    )


In [7]:
def tune_thresholds(y_true, probabilities):
    thresholds = {}
    threshold_scores = {}

    for index, aspect in enumerate(ASPECT_COLS):
        best_threshold = 0.5
        best_score = -1.0

        for threshold in np.arange(0.05, 0.951, 0.01):
            predictions = (
                probabilities[:, index] >= threshold
            ).astype(np.int8)
            score = f1_score(
                y_true[:, index],
                predictions,
                zero_division=0,
            )
            if score > best_score:
                best_score = score
                best_threshold = threshold

        thresholds[aspect] = round(float(best_threshold), 2)
        threshold_scores[aspect] = float(best_score)

    return thresholds, threshold_scores


def apply_thresholds(probabilities, thresholds):
    predictions = np.zeros_like(probabilities, dtype=np.int8)
    for index, aspect in enumerate(ASPECT_COLS):
        predictions[:, index] = (
            probabilities[:, index] >= thresholds[aspect]
        ).astype(np.int8)

    empty_rows = np.flatnonzero(predictions.sum(axis=1) == 0)
    for row in empty_rows:
        predictions[row, np.argmax(probabilities[row])] = 1
    return predictions


## Tune and save one classifier for each encoder

Only train and validation data are used here. The test split remains untouched for `03-mpnet_evaluate.ipynb`.


In [ ]:
all_grid_results = []
summary_rows = []

for encoder_name, encoder_path in ENCODER_PATHS.items():
    print(f"\n===== {encoder_name.upper()} =====")
    encoder = SentenceTransformer(str(encoder_path))

    X_train = encoder.encode(
        X_train_text,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    X_val = encoder.encode(
        X_val_text,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    results_df = run_grid_search(X_train, X_val)
    results_df.insert(0, "encoder", encoder_name)
    all_grid_results.append(results_df)

    best_row = results_df.iloc[0]
    best_params = {
        "C": float(best_row["C"]),
        "class_weight": (
            None
            if pd.isna(best_row["class_weight"])
            else best_row["class_weight"]
        ),
        "lr": float(best_row["lr"])
    }

    classifier = OneVsRestClassifier(
        ManualLogisticRegression(
            **best_params,
            max_iter=5000,
            random_state=42,
        ),
    )
    classifier.fit(X_train, y_train)
    val_probabilities = classifier.predict_proba(X_val)
    thresholds, threshold_scores = tune_thresholds(
        y_val,
        val_probabilities,
    )
    val_predictions = apply_thresholds(
        val_probabilities,
        thresholds,
    )

    classifier_dir = MODEL_DIR / "classifiers" / encoder_name
    classifier_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(classifier, classifier_dir / "model.joblib")

    metadata = {
        "encoder_name": encoder_name,
        "encoder_path": str(encoder_path.relative_to(PROJECT_ROOT)),
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "batch_size": BATCH_SIZE,
        "aspect_cols": ASPECT_COLS,
        "text_col": TEXT_COL,
        "parameters": best_params,
        "thresholds": thresholds,
        "validation_threshold_f1": threshold_scores,
        "validation_macro_f1": float(
            f1_score(
                y_val,
                val_predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "validation_micro_f1": float(
            f1_score(
                y_val,
                val_predictions,
                average="micro",
                zero_division=0,
            )
        ),
    }
    with (classifier_dir / "metadata.json").open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(metadata, file, indent=2)

    summary_rows.append({
        "encoder": encoder_name,
        "C": best_params["C"],
        "class_weight": best_params["class_weight"],
        'lr': best_params["lr"],

        "validation_macro_f1": metadata["validation_macro_f1"],
        "validation_micro_f1": metadata["validation_micro_f1"],
    })

grid_results_df = pd.concat(all_grid_results, ignore_index=True)
grid_results_df.to_csv(
    OUTPUT_DIR / "logistic_grid_results.csv",
    index=False,
)

summary_df = pd.DataFrame(summary_rows).sort_values(
    "validation_macro_f1",
    ascending=False,
)
summary_df.to_csv(
    OUTPUT_DIR / "classifier_validation_summary.csv",
    index=False,
)
summary_df



===== UNRETRAINED =====


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.45it/s]
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   27.3s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   46.9s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done  29 out of  32 | elapsed:  2.1min remaining:   12.7s
[Parallel(n_jobs=4)]: Done  32 out of  32 | elapsed:  2.1min finished



===== RETRAINED =====


Batches: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   17.9s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   30.8s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   49.6s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done  29 out of  32 | elapsed:  1.3min remaining:    8.2s
[Parallel(n_jobs=4)]: Done  32 out of  32 | elapsed:  1.4min finished


,encoder,C,class_weight,lr,validation_macro_f1,validation_micro_f1
1,retrained,0.05,balanced,0.01,0.830414,0.906907
0,unretrained,0.50,balanced,0.01,0.757820,0.848855
